In this notebook we generate `.json` files for a given subsample. These files will be used elsewhere to complete the analysis.

In [1]:
import json
import numpy as np
import pandas as pd

In [2]:
def write_paramfile(params):
    fname_out = f'/hpc/home/jem189/code/moore26/paramfiles/{params["instance_name"]}.json'
    with open(fname_out, 'w') as f:
        json.dump(params, f, indent=4)

# Load in beam corrections (computed by C. Popik)

In [3]:
ilc_bc = pd.read_csv('beamcorr/ilc_beamcorr.csv', header=None, index_col=0)
ilc_bc[ilc_bc<0] = 1
f090_bc = pd.read_csv('beamcorr/f090_beamcorr.csv', header=None, index_col=0)
f090_bc[f090_bc<0] = 1
f220_bc = pd.read_csv('beamcorr/f220_beamcorr.csv', header=None, index_col=0)
f220_bc[f220_bc<0] = 1

# Loop over all samples and maps

In [4]:
all_maps = {}

for f in ['f090', 'f150', 'f220']:
# for f in ['f090_srcfree', 'f150_srcfree', 'f220_srcfree']:
    # all_maps[f] = f'act-planck_dr6.02_coadd_AA_night_{f[0:4]}_map_srcfree'
    all_maps[f] = f'act-planck_dr6.02_coadd_AA_night_{f}_map'

for beta in [1.2, 1.4, 1.6]:
    for t in [10.7, 24.0]:
        all_maps[f'ilc_cib_{beta}_{t}'] = f'ilc_SZ_deproj_cib_{beta}_{t}_yy'
        all_maps[f'ilc_db_{beta}_{t}'] = f'ilc_SZ_deproj_cib_cibdBeta_{beta}_{t}_yy'

all_maps['ilc'] = 'ilc_SZ_yy'

## LRG

In [5]:
queries_lrg = {}
bins_lrg = {'L36':3.58e10, 'L48':4.78e10, 'L60':5.98e10, 'L79':7.86e10, 'L98':9.8e10, 'L36D':[3.58e10, 4.78e10], 'L48D':[4.78e10, 5.98e10], 'L60D':[5.98e10, 7.86e10], 'L79D':[7.86e10, 9.80e10]}
for b in bins_lrg.keys():
    if 'D' in b:
        queries_lrg[b] = f'lum > {bins_lrg[b][0]} and lum < {bins_lrg[b][1]}'
        queries_lrg[b+'_radio_clean'] = queries_lrg[b] + ' and `valid_radio_2.1` == 1'
        queries_lrg[b+'_radio_sources'] = queries_lrg[b] + ' and `valid_radio_2.1` == 0'

    else:
        queries_lrg[b] = f'lum > {bins_lrg[b]}'
        queries_lrg[b+'_radio_clean'] = queries_lrg[b] + ' and `valid_radio_2.1` == 1'
        queries_lrg[b+'_radio_sources'] = queries_lrg[b] + ' and `valid_radio_2.1` == 0'


# BGS

In [6]:
queries_bgs = {}
bins_bgs = {'M10':10, 'M10.25':10.25, 'M10.5':10.5, 'M10.75':10.75, 'M11':11, 'M11.25':11.25}
for b in bins_bgs.keys():
    queries_bgs[b] = f'logm > {bins_bgs[b]}'
    queries_bgs[b+'_radio_clean'] = queries_bgs[b] + ' and `valid_radio_2.7` == 1'
    queries_bgs[b+'_radio_sources'] = queries_bgs[b] + ' and `valid_radio_2.7` == 0'

# optical

In [7]:
queries_opt = {}
bins_opt = {'L5':5*0.79, 'L10':10*0.79, 'L20':20*0.79, 'L5D':[5*0.79, 10*0.79], 'L10D':[10*0.79, 20*0.79]}
for b in bins_opt.keys():
    if 'D' in b:
        queries_opt[b] = f'lum > {bins_opt[b][0]} and lum < {bins_opt[b][1]}'
        queries_opt[b+'_radio_clean'] = queries_opt[b] + ' and `valid_radio_2.4` == 1'
        queries_opt[b+'_radio_sources'] = queries_opt[b] + ' and `valid_radio_2.4` == 0'
    else:
        queries_opt[b] = f'lum > {bins_opt[b]}'
        queries_opt[b+'_radio_clean'] = queries_opt[b] + ' and `valid_radio_2.4` == 1'
        queries_opt[b+'_radio_sources'] = queries_opt[b] + ' and `valid_radio_2.4` == 0'

In [8]:
# freqs = {'f090_srcfree':90e9, 'f150_srcfree':150e9, 'f220_srcfree':220e9}
freqs = {'f090':90e9, 'f150':150e9, 'f220':220e9}
min_rs = {'lrg':0.6, 'bgs':0.7, 'opt':0.9}
max_rs = {'lrg':7.1, 'bgs':7.2, 'opt':7.4}
def make_params(map_id, sample_name, bin_name, cat_name, map_name, query):
    instance_name = f'{map_id}_{sample_name}_{bin_name}'
    min_r = min_rs[sample_name]
    max_r = max_rs[sample_name]
    r_rad_submap = np.deg2rad(10/60.) # radius of submap
    ap_path = f'/work/jem189/scratch/data/ApPhotoResults/{cat_name}/{cat_name}_{map_name}.hdf' # AP results file
    stack_file = f'/work/jem189/scratch/data/stacks/{cat_name}/{cat_name}_{map_name}_{bin_name}_stack.fits'
    if 'ilc' in map_id:
        freq = None
    else:
        freq = freqs[map_id]
    if '_radio_clean' in bin_name:
        shortname = bin_name[0:-12]
    elif '_radio_sources' in bin_name:
        shortname = bin_name[0:-14]
    else:
        shortname = bin_name
    params = {'sample_name':sample_name,
              'instance_name':instance_name,
              'cat_name':cat_name,
              'map_id':map_id,
              'freq':freq,
              'min_r':min_r,
              'max_r':max_r,
              'r_rad_submap':r_rad_submap,
              'map_name':map_name,
              'ap_path':ap_path,
              'query':query,
              'bin_name':bin_name,
              'stack_file':stack_file,
             'shortname':shortname}
    return params

# Make the files

In [10]:
# make list of filenames to use in slurm array job later
paramfile_names = [] 

In [11]:
for mkey, m in all_maps.items():
    for b in queries_lrg.keys():
        if 'ilc' in mkey:
            cat = 'ACT_DR6_DESI_Y1Iron_LRGs_11122025_valid_ilc_radioflagged'
        else:
            cat = 'ACT_DR6_DESI_Y1Iron_LRGs_10242025_valid_radioflagged'
        params = make_params(mkey, 'lrg', b, cat, m, queries_lrg[b])
        if 'ilc' in mkey:
            params["beam_corr"] = list(ilc_bc.loc[params['shortname']])
        elif 'f090' in mkey:
            params["beam_corr"] = list(f090_bc.loc[params['shortname']])
        elif 'f150' in mkey:
            params["beam_corr"] = list(np.ones_like(f090_bc.loc[params['shortname']]))
        elif 'f220' in mkey:
            params["beam_corr"] = list(f220_bc.loc[params['shortname']])
        paramfile_names.append(f'/hpc/home/jem189/code/moore26/paramfiles/{params["instance_name"]}.json')
        write_paramfile(params)

In [12]:
for mkey, m in all_maps.items():
    for b in queries_bgs.keys():
        cat = 'BGS_BRIGHT_full_noveto_vac_marvin_BGS_ACT_DR6_fixedTh2.10_delta_Ts_radioflagged'
        params = make_params(mkey, 'bgs', b, cat, m, queries_bgs[b])
        if 'ilc' in mkey:
            params["beam_corr"] = list(ilc_bc.loc[params['shortname']])
        elif 'f090' in mkey:
            params["beam_corr"] = list(f090_bc.loc[params['shortname']])
        elif 'f150' in mkey:
            params["beam_corr"] = list(np.ones_like(f090_bc.loc[params['shortname']]))
        elif 'f220' in mkey:
            params["beam_corr"] = list(f220_bc.loc[params['shortname']])
        paramfile_names.append(f'/hpc/home/jem189/code/moore26/paramfiles/{params["instance_name"]}.json')
        write_paramfile(params)

In [13]:
for mkey, m in all_maps.items():
    for b in queries_opt.keys():
        cat = 'iskay_RM_ls10_iron_v1_CGz_cat_310525_radioflagged'
        params = make_params(mkey, 'opt', b, cat, m, queries_opt[b])
        if 'ilc' in mkey:
            try:
                params["beam_corr"] = list(ilc_bc.loc[params['shortname']])
            except:
                params["beam_corr"] = list(ilc_bc.loc['L5']) # didn't get BC for disjoint bins
        elif 'f090' in mkey:
            try:
                params["beam_corr"] = list(f090_bc.loc[params['shortname']])
            except:
                params["beam_corr"] = list(f090_bc.loc['L5'])
        elif 'f150' in mkey:
            try:
                params["beam_corr"] = list(np.ones_like(f090_bc.loc[params['shortname']]))
            except:
                params["beam_corr"] = list(np.ones_like(f090_bc.loc['L5']))
        elif 'f220' in mkey:
            try:
                params["beam_corr"] = list(f220_bc.loc[params['shortname']])
            except:
                params["beam_corr"] = list(f220_bc.loc['L5'])
        paramfile_names.append(f'/hpc/home/jem189/code/moore26/paramfiles/{params["instance_name"]}.json')
        write_paramfile(params)

In [38]:
params

{'sample_name': 'opt',
 'instance_name': 'ilc_opt_L10D_radio_clean',
 'cat_name': 'iskay_RM_ls10_iron_v1_CGz_cat_310525_radioflagged',
 'map_id': 'ilc',
 'freq': None,
 'min_r': 0.9,
 'max_r': 7.4,
 'r_rad_submap': 0.002908882086657216,
 'map_name': 'ilc_SZ_yy',
 'ap_path': '/work/jem189/scratch/data/ApPhotoResults/iskay_RM_ls10_iron_v1_CGz_cat_310525_radioflagged/iskay_RM_ls10_iron_v1_CGz_cat_310525_radioflagged_ilc_SZ_yy.hdf',
 'query': 'lum > 7.9 and lum < 15.8 and `valid_radio_2.4` == 1',
 'bin_name': 'L10D_radio_clean',
 'stack_file': '/work/jem189/scratch/data/stacks/iskay_RM_ls10_iron_v1_CGz_cat_310525_radioflagged/iskay_RM_ls10_iron_v1_CGz_cat_310525_radioflagged_ilc_SZ_yy_L10D_radio_clean_stack.fits',
 'shortname': 'L10D',
 'beam_corr': [0.82669732,
  0.89595446,
  0.94288665,
  0.96736092,
  0.97946139,
  0.98604549,
  0.9899699,
  0.99246309,
  0.99414693,
  0.99532358,
  0.99617793,
  0.99680963,
  0.99729788]}

In [16]:
ap_paramfile_names = []

In [21]:
# Here I dump all the paramfiles needed for the ap_photo_config.txt file
# This makes it so I can easily run the sbatch script on all the catalog+map combinations
i = 1
for m in all_maps.keys():
    print(i, f'paramfiles/{m}_lrg_L36_radio_clean.json')
    i += 1
    print(i, f'paramfiles/{m}_bgs_M10_radio_clean.json')
    i += 1
    print(i, f'paramfiles/{m}_opt_L5_radio_clean.json')
    i += 1

1 paramfiles/f090_lrg_L36_radio_clean.json
2 paramfiles/f090_bgs_M10_radio_clean.json
3 paramfiles/f090_opt_L5_radio_clean.json
4 paramfiles/f150_lrg_L36_radio_clean.json
5 paramfiles/f150_bgs_M10_radio_clean.json
6 paramfiles/f150_opt_L5_radio_clean.json
7 paramfiles/f220_lrg_L36_radio_clean.json
8 paramfiles/f220_bgs_M10_radio_clean.json
9 paramfiles/f220_opt_L5_radio_clean.json
10 paramfiles/ilc_cib_1.2_10.7_lrg_L36_radio_clean.json
11 paramfiles/ilc_cib_1.2_10.7_bgs_M10_radio_clean.json
12 paramfiles/ilc_cib_1.2_10.7_opt_L5_radio_clean.json
13 paramfiles/ilc_db_1.2_10.7_lrg_L36_radio_clean.json
14 paramfiles/ilc_db_1.2_10.7_bgs_M10_radio_clean.json
15 paramfiles/ilc_db_1.2_10.7_opt_L5_radio_clean.json
16 paramfiles/ilc_cib_1.2_24.0_lrg_L36_radio_clean.json
17 paramfiles/ilc_cib_1.2_24.0_bgs_M10_radio_clean.json
18 paramfiles/ilc_cib_1.2_24.0_opt_L5_radio_clean.json
19 paramfiles/ilc_db_1.2_24.0_lrg_L36_radio_clean.json
20 paramfiles/ilc_db_1.2_24.0_bgs_M10_radio_clean.json
21 param

In [20]:
# Here I dump all the paramfiles needed for the config file for the stack_submaps and make_profile scripts
# This makes it so I can easily run the sbatch script on all the binned sample+map combinations
# 640 combinations total!!
# update: added paramfiles for sources WITH radio sources, so the output here is those paramfiles (saved to rs_paramfiles.txt)
i = 1
for p in paramfile_names:
    print(i, p)
    i+=1

1 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L36_radio_sources.json
2 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L48_radio_sources.json
3 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L60_radio_sources.json
4 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L79_radio_sources.json
5 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L98_radio_sources.json
6 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L36D_radio_sources.json
7 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L48D_radio_sources.json
8 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L60D_radio_sources.json
9 /hpc/home/jem189/code/moore26/paramfiles/f090_lrg_L79D_radio_sources.json
10 /hpc/home/jem189/code/moore26/paramfiles/f150_lrg_L36_radio_sources.json
11 /hpc/home/jem189/code/moore26/paramfiles/f150_lrg_L48_radio_sources.json
12 /hpc/home/jem189/code/moore26/paramfiles/f150_lrg_L60_radio_sources.json
13 /hpc/home/jem189/code/moore26/paramfiles/f150_lrg_L79_radio_sources.json
14 /hpc/home/jem1

In [11]:
# dump single frequency paramfiles for covariance computation
i = 1
for b in list(bins_lrg.keys()):
    for f in ['f090', 'f150', 'f220']:
        print(i, f'paramfiles/{f}_lrg_{b}.json')
        i+=1
        print(i, f'paramfiles/{f}_lrg_{b}_radio_clean.json')
        i+=1
for b in list(bins_bgs.keys()):
    for f in ['f090', 'f150', 'f220']:
        print(i, f'paramfiles/{f}_bgs_{b}.json')
        i+=1
        print(i, f'paramfiles/{f}_bgs_{b}_radio_clean.json')
        i+=1
for b in list(bins_opt.keys()):
    for f in ['f090', 'f150', 'f220']:
        print(i, f'paramfiles/{f}_opt_{b}.json')
        i+=1
        print(i, f'paramfiles/{f}_opt_{b}_radio_clean.json')
        i+=1

1 paramfiles/f090_lrg_L36.json
2 paramfiles/f090_lrg_L36_radio_clean.json
3 paramfiles/f150_lrg_L36.json
4 paramfiles/f150_lrg_L36_radio_clean.json
5 paramfiles/f220_lrg_L36.json
6 paramfiles/f220_lrg_L36_radio_clean.json
7 paramfiles/f090_lrg_L48.json
8 paramfiles/f090_lrg_L48_radio_clean.json
9 paramfiles/f150_lrg_L48.json
10 paramfiles/f150_lrg_L48_radio_clean.json
11 paramfiles/f220_lrg_L48.json
12 paramfiles/f220_lrg_L48_radio_clean.json
13 paramfiles/f090_lrg_L60.json
14 paramfiles/f090_lrg_L60_radio_clean.json
15 paramfiles/f150_lrg_L60.json
16 paramfiles/f150_lrg_L60_radio_clean.json
17 paramfiles/f220_lrg_L60.json
18 paramfiles/f220_lrg_L60_radio_clean.json
19 paramfiles/f090_lrg_L79.json
20 paramfiles/f090_lrg_L79_radio_clean.json
21 paramfiles/f150_lrg_L79.json
22 paramfiles/f150_lrg_L79_radio_clean.json
23 paramfiles/f220_lrg_L79.json
24 paramfiles/f220_lrg_L79_radio_clean.json
25 paramfiles/f090_lrg_L98.json
26 paramfiles/f090_lrg_L98_radio_clean.json
27 paramfiles/f150_lr

In [12]:
# dump samples for single-frequency deprojection
i = 1
for b in list(bins_lrg.keys()):
    print(i, f'lrg_{b}')
    i+=1
    print(i, f'lrg_{b}_radio_clean')
    i+=1
for b in list(bins_bgs.keys()):
    print(i, f'bgs_{b}')
    i+=1
    print(i, f'bgs_{b}_radio_clean')
    i+=1
for b in list(bins_opt.keys()):
    print(i, f'opt_{b}')
    i+=1
    print(i, f'opt_{b}_radio_clean')
    i+=1

1 lrg_L36
2 lrg_L36_radio_clean
3 lrg_L48
4 lrg_L48_radio_clean
5 lrg_L60
6 lrg_L60_radio_clean
7 lrg_L79
8 lrg_L79_radio_clean
9 lrg_L98
10 lrg_L98_radio_clean
11 lrg_L36D
12 lrg_L36D_radio_clean
13 lrg_L48D
14 lrg_L48D_radio_clean
15 lrg_L60D
16 lrg_L60D_radio_clean
17 lrg_L79D
18 lrg_L79D_radio_clean
19 bgs_M10
20 bgs_M10_radio_clean
21 bgs_M10.25
22 bgs_M10.25_radio_clean
23 bgs_M10.5
24 bgs_M10.5_radio_clean
25 bgs_M10.75
26 bgs_M10.75_radio_clean
27 bgs_M11
28 bgs_M11_radio_clean
29 bgs_M11.25
30 bgs_M11.25_radio_clean
31 opt_L5
32 opt_L5_radio_clean
33 opt_L10
34 opt_L10_radio_clean
35 opt_L20
36 opt_L20_radio_clean
37 opt_L5D
38 opt_L5D_radio_clean
39 opt_L10D
40 opt_L10D_radio_clean


In [92]:
# Here I dump all the paramfiles needed for the config file for the stack_submaps and make_profile scripts
# This makes it so I can easily run the sbatch script on all the binned sample+map combinations
# 640 combinations total!!
# update: added paramfiles for srcfree maps, so the output here is those paramfiles (saved to srcfree_paramfiles.txt)
i = 1
for p in paramfile_names:
    print(i, p)
    i+=1

1 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L36.json
2 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L36_radio_clean.json
3 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L36_radio_sources.json
4 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L48.json
5 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L48_radio_clean.json
6 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L48_radio_sources.json
7 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L60.json
8 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L60_radio_clean.json
9 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L60_radio_sources.json
10 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L79.json
11 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L79_radio_clean.json
12 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L79_radio_sources.json
13 /hpc/home/jem189/code/moore26/paramfiles/f090_srcfree_lrg_L98.

In [14]:
# dump single frequency paramfiles for covariance computation
i = 1
for b in list(bins_lrg.keys()):
    for f in ['f090_srcfree', 'f150_srcfree', 'f220_srcfree']:
        print(i, f'paramfiles/{f}_lrg_{b}.json')
        i+=1
        print(i, f'paramfiles/{f}_lrg_{b}_radio_clean.json')
        i+=1
for b in list(bins_bgs.keys()):
    for f in ['f090_srcfree', 'f150_srcfree', 'f220_srcfree']:
        print(i, f'paramfiles/{f}_bgs_{b}.json')
        i+=1
        print(i, f'paramfiles/{f}_bgs_{b}_radio_clean.json')
        i+=1
for b in list(bins_opt.keys()):
    for f in ['f090_srcfree', 'f150_srcfree', 'f220_srcfree']:
        print(i, f'paramfiles/{f}_opt_{b}.json')
        i+=1
        print(i, f'paramfiles/{f}_opt_{b}_radio_clean.json')
        i+=1

1 paramfiles/f090_srcfree_lrg_L36.json
2 paramfiles/f090_srcfree_lrg_L36_radio_clean.json
3 paramfiles/f150_srcfree_lrg_L36.json
4 paramfiles/f150_srcfree_lrg_L36_radio_clean.json
5 paramfiles/f220_srcfree_lrg_L36.json
6 paramfiles/f220_srcfree_lrg_L36_radio_clean.json
7 paramfiles/f090_srcfree_lrg_L48.json
8 paramfiles/f090_srcfree_lrg_L48_radio_clean.json
9 paramfiles/f150_srcfree_lrg_L48.json
10 paramfiles/f150_srcfree_lrg_L48_radio_clean.json
11 paramfiles/f220_srcfree_lrg_L48.json
12 paramfiles/f220_srcfree_lrg_L48_radio_clean.json
13 paramfiles/f090_srcfree_lrg_L60.json
14 paramfiles/f090_srcfree_lrg_L60_radio_clean.json
15 paramfiles/f150_srcfree_lrg_L60.json
16 paramfiles/f150_srcfree_lrg_L60_radio_clean.json
17 paramfiles/f220_srcfree_lrg_L60.json
18 paramfiles/f220_srcfree_lrg_L60_radio_clean.json
19 paramfiles/f090_srcfree_lrg_L79.json
20 paramfiles/f090_srcfree_lrg_L79_radio_clean.json
21 paramfiles/f150_srcfree_lrg_L79.json
22 paramfiles/f150_srcfree_lrg_L79_radio_clean.js